# 20.8 联邦学习 / Federated Learning (FedAvg)

**中文**:传统机器学习假设**所有数据都能汇总到一处**训练。但很多时候**数据不能集中**:①**隐私**(手机上的输入记录、健康数据不能上传);②**法规**(GDPR、医院病人数据不能出院);③**带宽**(海量边缘设备数据传不动)。**联邦学习(Federated Learning)** 解决这个矛盾:*"让数据留在本地,只传模型更新,协同训练一个共享模型。"* 是"**数据不动,模型动**"的范式。谷歌输入法、苹果、医疗联盟都在用。本节从零实现最经典的 **FedAvg** 算法,并揭示它最大的现实挑战——**非独立同分布(non-IID)**。
**English**: Traditional ML assumes **all data can be gathered in one place** to train. But often **data cannot be centralized**: ① **privacy** (phone keystrokes, health data can't be uploaded); ② **regulation** (GDPR, hospital patient data can't leave); ③ **bandwidth** (massive edge-device data can't be transferred). **Federated Learning** resolves this: *"keep data local, transmit only model updates, and collaboratively train a shared model."* The "**data stays put, the model moves**" paradigm. Google's keyboard, Apple, and medical consortia all use it. This section implements the classic **FedAvg** algorithm from scratch and reveals its biggest real-world challenge — **non-IID data**.

---

**中文**:**FedAvg(McMahan et al., 2017)** 算法极其简洁,每一轮(round):
**English**: **FedAvg (McMahan et al., 2017)** is elegantly simple; each round:
1. **服务器下发**:把当前全局模型参数发给一批客户端(手机/医院/机构)。
   **Server broadcasts**: send the current global model parameters to a batch of clients (phones/hospitals/organizations).
2. **本地训练**:每个客户端在**自己的本地数据**上做几步 SGD(**数据从不离开本地**)。
   **Local training**: each client runs a few SGD steps on **its own local data** (**data never leaves the device**).
3. **上传更新**:客户端只把**更新后的模型参数**(不是数据!)传回服务器。
   **Upload updates**: clients send back only the **updated model parameters** (not data!).
4. **服务器聚合**:把所有客户端的模型**按数据量加权平均**,得到新的全局模型。回到第 1 步。
   **Server aggregates**: **average the client models weighted by data size**, forming the new global model. Repeat.

$$w_{t+1}=\sum_{k=1}^{K}\frac{n_k}{n}\,w_{t+1}^{k}\quad(\text{按各客户端数据量 } n_k \text{ 加权平均})$$

**中文**:核心洞察:**多个本地模型的加权平均,近似于在合并数据上训练的效果**——但数据从没离开本地。隐私和协同两全。
**English**: Core insight: **a weighted average of local models approximates training on the pooled data** — yet data never leaves the device. Privacy and collaboration at once.

**中文**:**最大的现实挑战:non-IID(数据非独立同分布)**。理论上 FedAvg 假设每个客户端的数据是总体的随机样本(IID)。但现实里,**每个客户端的数据分布天差地别**——你手机上的照片和我的完全不同、A 医院的病种和 B 医院不同。此时各客户端的本地更新**方向相互冲突**,平均后互相抵消,收敛慢、精度掉。这是联邦学习最核心的难题。
**English**: **The biggest real-world challenge: non-IID data**. In theory FedAvg assumes each client's data is a random sample of the population (IID). But in reality, **each client's distribution differs wildly** — your phone's photos differ from mine, hospital A's cases differ from B's. Then clients' local updates **conflict in direction**, cancel when averaged, slowing convergence and dropping accuracy. This is federated learning's central difficulty.

> 💡 **面试速查 / Interview cheat-sheet（★★ 隐私/边缘AI必考）**
> **中文**:联邦学习=**数据不动、模型动**——数据留本地, 只传模型更新, 协同训练共享模型(隐私/法规/带宽驱动)。**FedAvg**:服务器下发→客户端本地SGD→上传参数→**按数据量加权平均**→重复。**最大挑战=non-IID**(各客户端分布差异大→本地更新冲突→收敛慢精度掉), 解法:**FedProx**(近端项限制本地偏离)、**SCAFFOLD**(控制变量纠偏)、动量。**其他挑战**:通信成本(压缩/量化梯度)、客户端掉线/异构(算力差异)、**隐私仍有风险**(梯度能反推数据→梯度泄漏攻击→需配 **差分隐私(20.9)+安全聚合**)。**vs 分布式训练**:分布式是"同一数据中心加速训练", 联邦是"数据分散且不能集中"。用途:手机输入法(Gboard)、医疗联盟、金融风控联合建模、IoT。
> **English**: Federated learning = **data stays put, the model moves** — data stays local, only model updates are sent, collaboratively training a shared model (driven by privacy/regulation/bandwidth). **FedAvg**: server broadcasts → clients do local SGD → upload parameters → **average weighted by data size** → repeat. **Biggest challenge = non-IID** (clients' distributions differ → local updates conflict → slow convergence, lower accuracy), fixes: **FedProx** (proximal term limiting local drift), **SCAFFOLD** (control variates to correct drift), momentum. **Other challenges**: communication cost (compress/quantize gradients), client dropout/heterogeneity (compute differences), **privacy still at risk** (gradients can reconstruct data → gradient-leakage attacks → combine with **differential privacy (20.9) + secure aggregation**). **vs distributed training**: distributed "speeds up training in one datacenter," federated "data is dispersed and can't be centralized." Uses: phone keyboards (Gboard), medical consortia, joint financial risk modeling, IoT.


In [ ]:

# ============================================================
# 把 MNIST 分给多个客户端 / partition MNIST across clients
# 中文:10 个客户端。IID:每个客户端拿到各类均匀的随机数据。non-IID:每个客户端只有1~2个数字类别
#       (模拟"每个人的数据分布天差地别")。数据永远留在客户端本地。
# English: 10 clients. IID: each gets a uniform random slice. non-IID: each has only 1-2 digit classes
#       (mimicking "everyone's distribution differs wildly"). Data always stays local.
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, os, time, copy, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)
from torchvision import datasets
root=os.path.expanduser("~/.cache/dsfs_cv")
mn=datasets.MNIST(root, train=True, download=False)
sub=np.random.permutation(len(mn.data))[:12000]
Xt=(mn.data.float()/255.).view(-1,784)[sub]; yt=mn.targets[sub]
te=datasets.MNIST(root, train=False, download=False)
Xtest=(te.data.float()/255.).view(-1,784)[:2000]; ytest=te.targets[:2000]
K=10                                                         # 客户端数 / number of clients
def partition(mode):
    if mode=="iid": return np.array_split(np.random.permutation(len(Xt)), K)   # 均匀随机 / uniform random
    return np.array_split(np.argsort(yt.numpy()), K)          # 按标签排序切→每客户端少数类 / sorted by label
parts_iid=partition("iid"); parts_noniid=partition("noniid")
print("IID 客户端0的类别分布 / IID client 0 labels:", np.bincount(yt[parts_iid[0]].numpy(),minlength=10))
print("non-IID 客户端0的类别分布 / non-IID client 0:", np.bincount(yt[parts_noniid[0]].numpy(),minlength=10),"← 只有个别类!")


**中文**:从零实现 FedAvg:客户端本地训练 + 服务器加权平均。对比三种情形:**集中式**(所有数据汇总,上界)、**FedAvg-IID**(数据均匀分布)、**FedAvg-non-IID**(数据分布差异大)。
**English**: Implement FedAvg from scratch: local client training + server weighted averaging. Compare three cases: **centralized** (all data pooled, the upper bound), **FedAvg-IID** (uniform distribution), **FedAvg-non-IID** (skewed distributions).


In [ ]:

# ============================================================
# FedAvg 从零实现 / FedAvg from scratch
# ============================================================
class Net(nn.Module):
    def __init__(s): super().__init__(); s.f=nn.Sequential(nn.Linear(784,128),nn.ReLU(),nn.Linear(128,10))
    def forward(s,x): return s.f(x)
def evaluate(state):
    m=Net(); m.load_state_dict(state)
    with torch.no_grad(): return (m(Xtest).argmax(1)==ytest).float().mean().item()

def local_train(global_state, idx, epochs=2, lr=0.05):       # 客户端本地训练(数据不出本地)/ local training
    m=Net(); m.load_state_dict(global_state); opt=torch.optim.SGD(m.parameters(),lr)
    for _ in range(epochs):
        perm=np.random.permutation(idx)
        for b in range(0,len(perm),64):
            bi=perm[b:b+64]; opt.zero_grad(); F.cross_entropy(m(Xt[bi]),yt[bi]).backward(); opt.step()
    return m.state_dict(), len(idx)

def fedavg(parts, rounds=15):
    glob=Net().state_dict(); curve=[]
    for r in range(rounds):
        client_states=[]; sizes=[]
        for k in range(K):                                   # 每个客户端各自本地训练 / each client trains locally
            st,n=local_train(copy.deepcopy(glob), parts[k]); client_states.append(st); sizes.append(n)
        total=sum(sizes)                                     # 服务器:按数据量加权平均 / weighted average
        glob={key: sum(sizes[k]/total*client_states[k][key] for k in range(K)) for key in glob}
        curve.append(evaluate(glob))
    return curve

t=time.time()
curve_iid=fedavg(parts_iid); curve_noniid=fedavg(parts_noniid)
# 集中式基线 / centralized baseline
m=Net(); opt=torch.optim.SGD(m.parameters(),0.05); cent_curve=[]
for e in range(15):
    perm=np.random.permutation(len(Xt))
    for b in range(0,len(perm),64):
        bi=perm[b:b+64]; opt.zero_grad(); F.cross_entropy(m(Xt[bi]),yt[bi]).backward(); opt.step()
    cent_curve.append((m(Xtest).argmax(1)==ytest).float().mean().item())
print(f"集中式(上界)/ centralized: {cent_curve[-1]:.3f}")
print(f"FedAvg IID(数据均匀)/ IID: {curve_iid[-1]:.3f}  ← 接近集中式!")
print(f"FedAvg non-IID(分布差异大): {curve_noniid[-1]:.3f}  ← 明显掉点(联邦学习头号难题)")
print(f"用时 {time.time()-t:.0f}s")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(14,5))
# ① 学习曲线 / learning curves
ax[0].plot(cent_curve,"k-",lw=2,label=f"集中式 centralized ({cent_curve[-1]:.2f})")
ax[0].plot(curve_iid,"o-",color="#4C72B0",ms=3,label=f"FedAvg IID ({curve_iid[-1]:.2f})")
ax[0].plot(curve_noniid,"o-",color="#C44E52",ms=3,label=f"FedAvg non-IID ({curve_noniid[-1]:.2f})")
ax[0].set_title("FedAvg:IID 接近集中式, non-IID 明显掉点 / non-IID hurts"); ax[0].set_xlabel("联邦轮次 round"); ax[0].set_ylabel("测试准确率"); ax[0].legend(fontsize=9)
# ② 各客户端的数据分布(non-IID)/ per-client label distribution (non-IID)
dist=np.array([np.bincount(yt[parts_noniid[k]].numpy(),minlength=10) for k in range(K)])
im=ax[1].imshow(dist,cmap="Blues",aspect="auto")
ax[1].set_title("non-IID:每个客户端只有个别数字类 / each client has few classes"); ax[1].set_xlabel("数字类别 digit"); ax[1].set_ylabel("客户端 client"); plt.colorbar(im,ax=ax[1],fraction=0.046,label="样本数")
plt.tight_layout(); plt.savefig("/tmp/adv08_viz.png",dpi=80); plt.show()
print("右图:non-IID 下每个客户端(行)只有1-2个数字类(列), 本地更新方向冲突→平均后收敛差")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **联邦学习真能"数据不动、模型动"**:FedAvg 在数据均匀(IID)时达到 0.87,**逼近集中式的 0.90**——也就是说,**数据从没离开客户端,却训练出了几乎和集中训练一样好的模型**。核心机制简单到优雅:多个本地模型的加权平均,近似于在合并数据上训练。这在隐私/法规不允许数据集中的场景(输入法、医疗、金融)是唯一可行的协同建模方式。
2. **non-IID 是联邦学习的头号拦路虎**:一旦每个客户端的数据分布差异巨大(本例每个客户端只有 1-2 个数字类),FedAvg 就从 0.87 崩到 0.58。原因看右图——每个客户端(行)只见过个别类别,它的本地更新会把模型往"只认自己那几个类"的方向拽,**十个客户端拽向十个不同方向,平均后互相抵消**,全局模型学不好。这不是 bug,而是联邦学习最核心、最难的现实问题(现实数据几乎都是 non-IID)。解法有 FedProx(加近端项限制本地偏离全局太远)、SCAFFOLD(用控制变量纠正漂移)等。
3. **诚实的隐私真相:联邦 ≠ 绝对安全**:很多人以为"数据没上传就绝对隐私",这是**危险的误解**。①**梯度会泄漏信息**——研究表明,从上传的梯度/模型更新能**反推出原始训练样本**(gradient leakage / deep leakage 攻击);②所以真正的隐私联邦学习**必须叠加**:**差分隐私**(给更新加噪声, 下节 20.9)+ **安全聚合**(服务器只能看到聚合结果、看不到单个客户端更新)+ 加密。③还有通信成本(每轮传整个模型很贵→梯度压缩/量化)、客户端异构与掉线等工程难题。**联邦学习是隐私的必要条件, 但远不是充分条件。**

**English**:
1. **Federated learning truly achieves "data stays put, the model moves"**: FedAvg reaches 0.87 on IID data, **approaching the centralized 0.90** — i.e., **data never left the clients, yet we trained a model almost as good as centralized training**. The mechanism is elegantly simple: a weighted average of local models approximates training on pooled data. Where privacy/regulation forbid centralizing data (keyboards, healthcare, finance), this is the only feasible collaborative modeling.
2. **non-IID is federated learning's #1 roadblock**: once clients' distributions differ greatly (here each client has only 1-2 digit classes), FedAvg collapses from 0.87 to 0.58. See the right plot — each client (row) has seen only a few classes, so its local update pulls the model toward "recognizing only my classes," and **ten clients pull in ten different directions, canceling when averaged**, so the global model learns poorly. Not a bug but federated learning's central, hardest real-world problem (real data is almost always non-IID). Fixes include FedProx (a proximal term limiting local drift from the global), SCAFFOLD (control variates to correct drift), etc.
3. **Honest privacy truth: federated ≠ absolutely secure**: many think "data wasn't uploaded, so it's absolutely private" — a **dangerous misconception**. ① **Gradients leak information** — research shows original training samples can be **reconstructed from uploaded gradients/updates** (gradient-leakage / deep-leakage attacks); ② so genuinely private federated learning **must add**: **differential privacy** (noise on updates, 20.9 next) + **secure aggregation** (the server sees only the aggregate, not individual updates) + encryption; ③ plus communication cost (sending the whole model each round is expensive → gradient compression/quantization), client heterogeneity and dropout. **Federated learning is a necessary but far from sufficient condition for privacy.**

> 💼 **实战视角 / Practical angle**
> **中文**:联邦学习的落地:①**手机端**(谷歌 Gboard 输入预测、语音、Apple)——模型在你手机上用你的数据训练, 只上传更新;②**医疗联盟**(多家医院联合建模而不共享病人数据, NVIDIA Clara/FLARE);③**金融风控**(多家银行联合反欺诈, 数据不出行);④IoT/边缘。工程栈:`Flower`、`FedML`、`TensorFlow Federated`、`PySyft`。落地要点:①**先评估 non-IID 程度**, 严重就用 FedProx/SCAFFOLD/个性化联邦(每客户端微调);②**通信是瓶颈**——减少轮数、压缩更新、只选部分客户端;③**隐私必须叠加 DP + 安全聚合**(别以为不传数据就安全);④客户端异构/掉线要容错。面试金句:*"联邦学习让数据留本地、只传模型更新, FedAvg 按数据量加权平均本地模型≈在合并数据上训练; 头号难题是 non-IID(客户端分布差异→更新冲突→掉点, 用 FedProx/SCAFFOLD); 且联邦≠绝对隐私, 梯度可反推数据, 需叠加差分隐私和安全聚合。"*
> **English**: Federated learning in practice: ① **on-device** (Google Gboard next-word prediction, voice, Apple) — the model trains on your phone with your data, uploading only updates; ② **medical consortia** (hospitals jointly modeling without sharing patient data, NVIDIA Clara/FLARE); ③ **financial risk** (banks jointly fighting fraud, data stays in-house); ④ IoT/edge. Stack: `Flower`, `FedML`, `TensorFlow Federated`, `PySyft`. Deployment keys: ① **first assess the non-IID degree**, use FedProx/SCAFFOLD/personalized federated (per-client fine-tuning) if severe; ② **communication is the bottleneck** — fewer rounds, compressed updates, sample a subset of clients; ③ **privacy must add DP + secure aggregation** (don't assume not-sending-data is safe); ④ tolerate client heterogeneity/dropout. Interview line: *"Federated learning keeps data local and sends only model updates; FedAvg's data-weighted average of local models ≈ training on pooled data; the #1 challenge is non-IID (clients' distributions differ → conflicting updates → accuracy drop, use FedProx/SCAFFOLD); and federated ≠ absolute privacy — gradients can reconstruct data, so add differential privacy and secure aggregation."*

---
### 小结 / Summary
- **中文**:联邦学习=数据不动、模型动; FedAvg=本地SGD+服务器按数据量加权平均, IID 时接近集中式。
- **English**: Federated learning = data stays put, the model moves; FedAvg = local SGD + server data-weighted averaging, near-centralized when IID.
- **中文**:头号难题 non-IID(客户端分布差异→更新冲突→掉点, 0.87→0.58); 解法 FedProx/SCAFFOLD/个性化联邦。
- **English**: #1 challenge non-IID (clients differ → conflicting updates → drop, 0.87→0.58); fixes FedProx/SCAFFOLD/personalized federated.
- **中文**:联邦≠绝对隐私(梯度可反推数据), 需叠加差分隐私(下节)+安全聚合; 通信成本也是瓶颈。
- **English**: Federated ≠ absolute privacy (gradients can reconstruct data), needs differential privacy (next) + secure aggregation; communication cost is also a bottleneck.
